# Data Cleaning Notebook - Sanjana, Lauren, and Yoojeong

We identified which data sets we wanted to use which are listed below:

From IMDB
* `name.basics`
* `title.basics`
* `title.principals`
* `title.ratings`
  
From Kaggle
* `mpst_full_data`

This amounts to approximately 1.36 GB.


## Step 1. Ingesting the data (title.basics, title.ratings, name.basics) into dataframes

In [ ]:
#pip install duckdb

In [ ]:
import pandas as pd

# 1. title.basics: The Master List of movies
# Columns in this data set - tconst (identifier), titleType, primaryTitle, originalTitle, isAdult, startYear, endYear, runtimeMinutes, genres (seperated by commas)
basics = pd.read_csv('title.basics.tsv.gz',
                     sep='\t',
                     compression='gzip',
                     low_memory=False,
                     na_values='\\N')

# 2. title.ratings: The movie ratings
# Columns in this data set - tconst (identifier),averageRating, numVotes
ratings = pd.read_csv('title.ratings.tsv.gz',
                      sep='\t',
                      compression='gzip',
                      na_values='\\N')

# 3. name.basics: names of people in the movies
names = pd.read_csv('name.basics.tsv.gz',
                    sep='\t',
                    usecols=['nconst', 'primaryName'],
                    na_values='\\N')


In [3]:
prefilter_principal = pd.merge(basics, ratings, on='tconst')
prefilter_principal_id = set(prefilter_principal['tconst'])

import duckdb

principals = duckdb.sql(f"""
    SELECT tconst, nconst, category
    FROM read_csv('title.principals.tsv.gz',
                  delim='\t',
                  nullstr='\\N')
    WHERE category IN ('director', 'actor', 'actress')
      AND tconst IN ({','.join(f"'{i}'" for i in prefilter_principal_id)})
""").df()

# merge names and principals
principals = pd.merge(principals, names, on='nconst')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## Step 2: Cleaning dataframes - imputing, converting into correct datatypes

### A. Cleaning dataframe - **Basics**

In [4]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer


#1) drop endYear
basics.drop(columns=['endYear'], inplace=True)

#2) change column names
basics.rename(columns={
    'runtimeMinutes': 'Runtime(min)',
    'tconst': 'Movie ID',
    'isAdult': 'Adult-content',
    'startYear': 'Release Year',
    'titleType': 'Type',
    'genres': 'Genre'},
inplace=True)

#3) change genres to Genre and split into list
basics['Genre'] = basics['Genre'].str.split(',')

#4) change Adult-content to boolean(Yes/No)
basics['Adult-content'] = basics['Adult-content'].map({0: 'No', 1: 'Yes'})

#5) merge title
basics['Title'] = np.where(
    basics['originalTitle'].isna() | (basics['originalTitle'] == '') | (basics['primaryTitle'] == basics['originalTitle']),
    basics['primaryTitle'],
    basics['primaryTitle'] + ' (' + basics['originalTitle'] + ')'
)
basics.drop(columns=['primaryTitle', 'originalTitle'], inplace=True)

#reorder columns
basics = basics[['Movie ID', 'Title', 'Type', 'Adult-content', 'Release Year', 'Runtime(min)', 'Genre']]

#6) change Release Year to int and fill missing values with median
basics['Release Year'] = pd.to_numeric(basics['Release Year'], errors='coerce')
imputer = SimpleImputer(strategy='median')
basics[['Release Year']] = imputer.fit_transform(basics[['Release Year']])
basics['Release Year'] = basics['Release Year'].round().astype('Int64')


#7) change Runtime(min) to int and fill missing values with median
basics['Runtime(min)'] = pd.to_numeric(basics['Runtime(min)'], errors='coerce')
imputer = SimpleImputer(strategy='median')
basics[['Runtime(min)']] = imputer.fit_transform(basics[['Runtime(min)']])
basics['Runtime(min)'] = basics['Runtime(min)'].round().astype('Int64')

#8) drop rows with missing Title
basics = basics.dropna(subset=['Title'])

#9) Drop rows except for movies and tvMovies
basics = basics[basics['Type'].isin(['movie', 'tvMovie'])]

basics.tail()



,Movie ID,Title,Type,Adult-content,Release Year,Runtime(min),Genre
12347215,tt9916680,De la ilusión al desconcierto: cine colombiano...,movie,No,2007,100,[Documentary]
12347220,tt9916692,Czlowiek bez twarzy,tvMovie,No,2015,66,[Drama]
12347227,tt9916706,Dankyavar Danka,movie,No,2013,30,[Comedy]
12347237,tt9916730,6 Gunn,movie,No,2017,116,[Drama]
12347247,tt9916754,Chico Albuquerque - Revelações,movie,No,2013,49,[Documentary]


In [5]:
# copy for imputation of missing Genre values
b = basics.copy()

# Convert Genre (list -> string)
b['Genre_str'] = b['Genre'].apply(lambda x: ','.join(x) if isinstance(x, list) else x)

# Missing Genre mask
missing = b['Genre_str'].isna() | (b['Genre_str'].astype(str).str.strip() == '')

# create additional columns for imputation
b['Release Year'] = pd.to_numeric(b['Release Year'], errors='coerce')
b['Runtime(min)'] = pd.to_numeric(b['Runtime(min)'], errors='coerce')
b['decade'] = (b['Release Year'] // 10) * 10
b['runtime_bin'] = pd.cut(
    b['Runtime(min)'],
    bins=[0, 20, 30, 45, 60, 90, 120, 150, 1000],
    labels=['<=20', '21-30', '31-45', '46-60', '61-90', '91-120', '121-150', '150+']
)

# Using only rows with known Genre for imputation
known = b.loc[~missing].copy()

# 1) Fill by (Type, decade, runtime_bin)
mode1 = (
    known.groupby(['Type', 'decade', 'runtime_bin'])['Genre_str']
    .agg(lambda s: s.mode().iat[0] if not s.mode().empty else np.nan)
    .reset_index(name='fill1')
)
b = b.merge(mode1, on=['Type', 'decade', 'runtime_bin'], how='left')
m = b['Genre_str'].isna() | (b['Genre_str'].astype(str).str.strip() == '')
b.loc[m, 'Genre_str'] = b.loc[m, 'fill1']
b.drop(columns=['fill1'], inplace=True)

# 2) Fallback by (Type, decade)
mode2 = (
    known.groupby(['Type', 'decade'])['Genre_str']
    .agg(lambda s: s.mode().iat[0] if not s.mode().empty else np.nan)
    .reset_index(name='fill2')
)
b = b.merge(mode2, on=['Type', 'decade'], how='left')
m = b['Genre_str'].isna() | (b['Genre_str'].astype(str).str.strip() == '')
b.loc[m, 'Genre_str'] = b.loc[m, 'fill2']
b.drop(columns=['fill2'], inplace=True)

# 3) Fallback by Type
mode3 = (
    known.groupby('Type')['Genre_str']
    .agg(lambda s: s.mode().iat[0] if not s.mode().empty else np.nan)
    .reset_index(name='fill3')
)
b = b.merge(mode3, on='Type', how='left')
m = b['Genre_str'].isna() | (b['Genre_str'].astype(str).str.strip() == '')
b.loc[m, 'Genre_str'] = b.loc[m, 'fill3']
b.drop(columns=['fill3'], inplace=True)

# Convert back to list format; keep unresolved as NaN
b['Genre'] = b['Genre_str'].apply(
    lambda s: s.split(',') if isinstance(s, str) and s.strip() != '' else np.nan
)

# Clean up
basics = b.drop(columns=['Genre_str', 'decade', 'runtime_bin'])

# Drop remaining NaN
basics = basics.dropna(subset=['Genre'])


/var/folders/mj/9hd_fv09537bn8xmcfz_k_kr0000gn/T/ipykernel_76477/1665279720.py:25: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  known.groupby(['Type', 'decade', 'runtime_bin'])['Genre_str']


In [6]:

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

b = basics.copy()

# Make a simple text feature from Genre list
b['Genre_text'] = b['Genre'].apply(lambda x: ','.join(x) if isinstance(x, list) else x)

# Split known vs missing Adult-content rows
known = b[b['Adult-content'].isin(['Yes', 'No'])].copy()
missing = b[~b['Adult-content'].isin(['Yes', 'No'])].copy()

# Set X, y
X = known[['Type', 'Release Year', 'Runtime(min)', 'Genre_text']]
y = known['Adult-content']

# split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=617, stratify=y
)

# Testing
test = ColumnTransformer(
    transformers=[
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore'))
        ]), ['Type', 'Genre_text']),
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), ['Release Year', 'Runtime(min)'])
    ]
)

# Model 1
model1 = Pipeline([
    ('prep', test),
    ('model', LogisticRegression(max_iter=1000, class_weight='balanced'))
])

# Train and test
model1.fit(X_train, y_train)
pred_test = model1.predict(X_test)

# Fill missing Adult-content
if len(missing) > 0:
    X_missing = missing[['Type', 'Release Year', 'Runtime(min)', 'Genre_text']]
    proba = model1.predict_proba(X_missing)
    classes = model1.classes_
    yes_idx = np.where(classes == 'Yes')[0][0]
    p_yes = proba[:, yes_idx]

    high = 0.95
    fill_yes = missing.index[p_yes >= high]
    fill_no = missing.index[p_yes <= (1 - high)]

    b.loc[fill_yes, 'Adult-content'] = 'Yes'
    b.loc[fill_no, 'Adult-content'] = 'No'

# Final cleanup
basics = b.drop(columns=['Genre_text'])


In [7]:
total_nan = basics.isna().sum().sum()
print("Total NaN in basics:", total_nan)

nan_by_col = basics.isna().sum()
print(nan_by_col)
basics.tail()

Total NaN in basics: 0
Movie ID         0
Title            0
Type             0
Adult-content    0
Release Year     0
Runtime(min)     0
Genre            0
dtype: int64


,Movie ID,Title,Type,Adult-content,Release Year,Runtime(min),Genre
894093,tt9916680,De la ilusión al desconcierto: cine colombiano...,movie,No,2007,100,[Documentary]
894094,tt9916692,Czlowiek bez twarzy,tvMovie,No,2015,66,[Drama]
894095,tt9916706,Dankyavar Danka,movie,No,2013,30,[Comedy]
894096,tt9916730,6 Gunn,movie,No,2017,116,[Drama]
894097,tt9916754,Chico Albuquerque - Revelações,movie,No,2013,49,[Documentary]


### B. Cleaning data - **principals**

In [8]:
# rename columns - principals
principals.rename(columns= {
    'tconst': 'Movie ID',
    'nconst': 'Person ID',
    'category': 'Job title',
    'primaryName': 'Name'},
inplace=True)

principals.head()

,Movie ID,Person ID,Job title,Name
0,tt0018036,nm0863833,actor,Rin Tin Tin
1,tt0018036,nm0730165,actor,Jason Robards Sr.
2,tt0018036,nm0272450,actress,Helen Ferguson
3,tt0018036,nm0588928,actress,Mary Louise Miller
4,tt0018036,nm0193354,actor,Jack Curtis


In [9]:
# Check for NaN values

total_nan = principals.isna().sum().sum()
print("Total NaN in principals:", total_nan)

nan_by_col = principals.isna().sum()
print(nan_by_col)

Total NaN in principals: 0
Movie ID     0
Person ID    0
Job title    0
Name         0
dtype: int64


### C. Cleaning data - **rating**

In [10]:

#rename columns
ratings.rename(columns={
    'tconst': 'Movie ID',
    'averageRating': 'Average Rating',
    'numVotes': 'Number of Votes'},
inplace=True)


ratings.head()

,Movie ID,Average Rating,Number of Votes
0,tt0000001,5.7,2198
1,tt0000002,5.5,310
2,tt0000003,6.5,2304
3,tt0000004,5.1,196
4,tt0000005,6.2,3035


In [11]:
#Vote-Adjusted Movie Rating (Bayesian Weighted Rating)

# R: movie's average rating, v: number of votes
R = ratings['Average Rating']
v = ratings['Number of Votes']

# C: global mean rating across all movies
C = R.mean()

# m: minimum votes threshold (example: 75th percentile) - 0.75 can be adjusted based on how strict we want to be
m = v.quantile(0.75)

# Weighted rating formula
ratings['Weighted Rating'] = (v / (v + m)) * R + (m / (v + m)) * C

# Round Weighted Rating to 1 decimal place
ratings['Weighted Rating'] = ratings['Weighted Rating'].round(1)

#change order of columns
ratings = ratings[['Movie ID', 'Weighted Rating', 'Number of Votes', 'Average Rating']]

#drop columns
ratings.drop(columns=['Average Rating', 'Number of Votes'], inplace=True)

# Check top results
ratings.head()

,Movie ID,Weighted Rating
0,tt0000001,5.8
1,tt0000002,5.9
2,tt0000003,6.5
3,tt0000004,5.7
4,tt0000005,6.2


## Step 3: Uploading and cleaning Kaggle movie plotline csv  

In [12]:
plotline = pd.read_csv('mpst_full_data.csv')
plotline = plotline[['imdb_id', 'plot_synopsis']]
plotline.rename(columns={'imdb_id': 'Movie ID', 'plot_synopsis': 'Plot Synopsis'}, inplace=True)

plotline['Plot Synopsis'] = plotline['Plot Synopsis'].str.replace(r'[^a-zA-Z0-9\s]', '', regex=True) #remove any symbols that aren't letters or numbe

plotline.head()

,Movie ID,Plot Synopsis
0,tt0057603,Note this synopsis is for the orginal Italian ...
1,tt1733125,Two thousand years ago Nhagruul the Foul a sor...
2,tt0033045,Matuscheks a gift store in Budapest is the wor...
3,tt0113862,Glenn Holland not a morning person by anyones ...
4,tt0086250,In May 1980 a Cuban man named Tony Montana Al ...


## Step 4: Finalizing movies dataframe with all necessary columns

In [13]:
# Restrict all dataframes to movies that exist in basics before merging.
# This avoids inflating row counts with orphaned records — ratings, principals,
# or plotlines that reference movies not in our core dataset.
base_ids = set(basics['Movie ID'])

ratings = ratings[ratings['Movie ID'].isin(base_ids)].copy()
principals = principals[principals['Movie ID'].isin(base_ids)].copy()
plotline = plotline[plotline['Movie ID'].isin(base_ids)].copy()

In [14]:
movies = pd.merge(basics, ratings, on='Movie ID')
movies = pd.merge(movies, plotline, on='Movie ID')
movie_ids = set(movies['Movie ID'])

movies.head()


,Movie ID,Title,Type,Adult-content,Release Year,Runtime(min),Genre,Weighted Rating,Plot Synopsis
0,tt0002130,Dante's Inferno (L'inferno),movie,No,1911,71,"[Adventure, Drama, Fantasy]",7.0,The exhumation of Lizzie Siddals desiccated bo...
1,tt0003419,The Student of Prague (Der Student von Prag),movie,No,1913,83,"[Drama, Fantasy, Horror]",6.4,Being praised as the finest fencer in his Univ...
2,tt0003489,The Last Days of Pompeii (Gli ultimi giorni di...,movie,No,1913,107,"[Adventure, Drama]",6.2,In Pompeii 79AD Glaucus and Jone are in love w...
3,tt0004022,Julius Caesar (Cajus Julius Caesar),movie,No,1914,112,"[Drama, History]",6.7,The play opens with the commoners of Rome cele...
4,tt0004099,"His Majesty, the Scarecrow of Oz",movie,No,1914,59,"[Adventure, Comedy, Family]",5.6,King Krewl Raymond Russell is a cruel dictator...


In [15]:
# Check for NaN values

total_nan = movies.isna().sum().sum()
print("Total NaN in movies:", total_nan)

nan_by_col = movies.isna().sum()
print(nan_by_col)

Total NaN in movies: 0
Movie ID           0
Title              0
Type               0
Adult-content      0
Release Year       0
Runtime(min)       0
Genre              0
Weighted Rating    0
Plot Synopsis      0
dtype: int64


# Export the dataframes to CSV files

In [ ]:
basics.to_csv('basics.csv', index=False)
principals.to_csv('principals.csv', index=False)
ratings.to_csv('ratings.csv', index=False)
names.to_csv('names.csv', index=False)
plotline.to_csv('plotline.csv', index=False)
movies.to_csv('movies.csv', index=False)

